# K06_02_GridSearchCV_fuer_Entscheidungsbaeume



# Hyperparameter Suche mit `**GridSearchCV**`

# Bibliotheken und Daten laden

## Der Breast Cancer Datensatz

Ein klassischer ML-Datensatz aus der medizinischen Diagnostik, der in scikit-learn direkt eingebaut ist:

```python
from sklearn.datasets import load_breast_cancer
data = load_breast_cancer()
```

---

### Was steckt drin?

| Eigenschaft | Wert |
|---|---|
| Datenpunkte | 569 Tumorproben |
| Features | 30 numerische Merkmale |
| Klassen | 2 (binär) |
| Klasse 0 | malignant – bösartig (212 Fälle) |
| Klasse 1 | benign – gutartig (357 Fälle) |

---

### Woher kommen die Daten?

Jede Zeile ist eine **Gewebeprobe** eines Brusttumors. Ein Arzt hat Zellen unter dem Mikroskop vermessen. Die 30 Features beschreiben die Geometrie der Zellkerne, z. B.:

- `mean radius` – durchschnittlicher Radius der Zellen
- `mean texture` – Oberflächenstruktur
- `mean concavity` – Einbuchtungen am Zellrand
- `mean symmetry` – Symmetrie der Zellen

---

### Die Aufgabe für das Modell

```
30 Messwerte einer Probe  →  Modell  →  bösartig oder gutartig?
```

Das ist ein **binäres Klassifikationsproblem** — genau das, wofür logistische Regression und Entscheidungsbäume ideal sind.

---


In [1]:
from sklearn.datasets import load_breast_cancer
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score

# Daten laden
X, y = load_breast_cancer(return_X_y=True)
X.shape, y.shape

((569, 30), (569,))

# Trainings- und Testdaten erstellen

In [2]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Classifier für Entscheidungsbäume erstellen und evaluieren (im Default)

In [3]:
tree_clf_default = DecisionTreeClassifier(random_state=42)
tree_clf_default.fit(X_train, y_train)

y_pred_default = tree_clf_default.predict(X_test)
accuracy_score(y_test, y_pred_default)

0.9122807017543859

# `GridSeachCV` durchführen und evaluieren


## Die Parameter des `param_grid` erklärt

Alle vier Parameter steuern, wie groß und komplex der Entscheidungsbaum werden darf.

---

### `max_depth` – Maximale Tiefe des Baums

Wie viele Ebenen darf der Baum haben?

```
max_depth=2:          max_depth=4:
      [A]                   [A]
     /   \                 /   \
   [B]   [C]            [B]   [C]
   (fertig)            /   \   /  \
                     [D][E][F][G]
                     / \ ...
                  (noch tiefer)
```

Ein tiefer Baum lernt sehr spezifische Regeln → Gefahr: Overfitting.
Ein flacher Baum lernt nur grobe Regeln → Gefahr: Underfitting.

---

### `max_leaf_nodes` – Maximale Anzahl Blätter

Blätter sind die **Endknoten** des Baums — dort wird die finale Klasse ausgegeben. Dieser Parameter begrenzt direkt, wie viele verschiedene Entscheidungen der Baum treffen kann:

```
max_leaf_nodes=2:  nur 2 mögliche Endergebnisse  →  sehr einfach
max_leaf_nodes=9:  bis zu 9 mögliche Endergebnisse →  differenzierter
```

Wirkt ähnlich wie `max_depth`, aber direkter — man begrenzt die Blätter, nicht die Ebenen.

---

### `min_samples_split` – Mindestgröße zum Aufteilen

Wie viele Datenpunkte müssen in einem Knoten liegen, damit er **überhaupt aufgeteilt** werden darf?

```
min_samples_split=2:   schon ab 2 Punkten wird aufgeteilt  →  Baum wird sehr fein
min_samples_split=20:  erst ab 20 Punkten wird aufgeteilt  →  Baum bleibt grober
```

Kleine Werte → der Baum teilt auch winzige Gruppen noch weiter auf → Overfitting-Risiko.  
Große Werte → der Baum hört früher auf zu teilen → robustere, allgemeinere Regeln.

---

### `min_samples_leaf` – Mindestgröße eines Blatts

Wie viele Datenpunkte muss ein **Endknoten (Blatt) mindestens** enthalten?

```
min_samples_leaf=1:   ein Blatt darf für nur 1 Datenpunkt existieren  →  sehr speziell
min_samples_leaf=10:  ein Blatt muss mindestens 10 Punkte repräsentieren →  stabiler
```

Das verhindert, dass der Baum einzelne Ausreißer als eigene Regel lernt.

---

### Zusammenfassung

| Parameter | Steuert | Kleiner Wert | Großer Wert |
|---|---|---|---|
| `max_depth` | Tiefe des Baums | komplex, Overfitting | einfach, Underfitting |
| `max_leaf_nodes` | Anzahl Endknoten | wenig Entscheidungen | mehr Entscheidungen |
| `min_samples_split` | Wann wird geteilt? | feiner, komplexer | grober, robuster |
| `min_samples_leaf` | Mindestgröße Blatt | speziell, instabil | allgemein, stabil |

> **Merksatz:** Alle vier Parameter sind **Komplexitätsbremsen**. GridSearchCV probiert alle Kombinationen durch und findet heraus, welche Bremskombination den besten Score auf den Testdaten liefert.

In [4]:
# Hyperparameter-Gitter
param_grid = {
    'max_leaf_nodes': list(range(2, 10)),
    "max_depth": list(range(1, 10)),
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 5, 10]
}

# GridSearchCV
grid = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid.fit(X_train, y_train)
print("Best Estimator:", grid.best_estimator_)
print("Best Parameters:", grid.best_params_)
print("Best Score:", grid.best_score_)

Best Estimator: DecisionTreeClassifier(max_depth=3, max_leaf_nodes=5, random_state=42)
Best Parameters: {'max_depth': 3, 'max_leaf_nodes': 5, 'min_samples_leaf': 1, 'min_samples_split': 2}
Best Score: 0.9362637362637362


# Beste Hyperparameter auf Testdaten evaluieren

In [5]:
y_pred = grid.predict(X_test)
accuracy_score(y_test, y_pred)

0.9385964912280702